# 01 — Development-cohort analyses

Thin driver around `scripts/run_development.py`. All analysis logic lives in the tested
`hfrevisit` package; this notebook only runs the stages and displays the outputs.

| Stage | Output | Manuscript |
|---|---|---|
| `table1` | `table1_cohort_by_outcome.csv` | Table 1 |
| `replication` | `replication_summary.csv`, `replication_delta_mcc_S1_minus_S2.csv` | Table S1 |
| `leakage` | `leakage_*`, `fig1_time_leakage` | Fig. 1, Table S2 |
| `survival` | `survival_cv_summary.csv`, `table2_cox_hazard_ratios.csv`, `fig2*`, `fig3*` | Table 2, Figs 2–3 |
| `explain` | `importance_*`, `fig4*` | Fig. 4 |
| `fairness` | `table3_subgroup_cindex.csv` | Table 3 |
| `simulation` | `simulation_summary.csv`, `fig_s_simulation_leakage` | Fig. S1 |
| `summary` | `results/RESULTS_SUMMARY.md` | every number in the text |
| `export` | `models/development_survival_models.pkl` | input to notebook 02 |

Runtime: about 2-4 h on a free Colab CPU in total, split over six cells.
The committed `results/` folder already contains the outputs of this notebook.

In [ ]:
# Setup (Colab or local). Outputs go to HF_PROJECT_ROOT, a Google Drive folder on Colab,
# so they survive a disconnect. Re-running this cell is safe.
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
REPO_DIR = '/content/HF-Survival-Analysis-Leakage-Revisit'
def sh(*cmd):
    r = subprocess.run(list(cmd), capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError('Command failed: ' + ' '.join(cmd))
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['HF_PROJECT_ROOT'] = '/content/drive/MyDrive/HF_Revisit_Project'
    if not os.path.exists(REPO_DIR):
        sh('git', 'clone', '-q', '-b', 'claude/ecstatic-wright-61dz0b', 'https://github.com/elifnuroksuzz/HF-Survival-Analysis-Leakage-Revisit.git', REPO_DIR)
    os.chdir(REPO_DIR)
    sh('git', 'fetch', '-q', 'origin', 'claude/ecstatic-wright-61dz0b')
    sh('git', 'reset', '-q', '--hard', 'origin/claude/ecstatic-wright-61dz0b')   # always the latest code
    sh(sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[deep]')
sys.path.insert(0, 'src')
from hfrevisit import config as C
P = C.get_paths()
print('Code version:', subprocess.run(['git', 'log', '-1', '--format=%h %cd', '--date=short'],
                                      capture_output=True, text=True).stdout.strip())
print('Outputs are written to:', P.root)

## Run the analyses

Run the cells **in order**. Each cell is one group of stages and saves its tables and
figures to Drive as soon as it finishes, so if Colab disconnects you only need to re-run
the setup cell and then continue from the cell that was interrupted. Approximate times are
for a free Colab CPU runtime.

In [ ]:
# 1/6  Download data (SHA-256 checked), Table 1, replication, leakage experiments (~30-60 min)
!python scripts/run_development.py --download --stages table1 replication leakage

In [ ]:
# 2/6  Survival models (repeated CV), calibration, decision curves, subgroups, explanation (~30-60 min)
!python scripts/run_development.py --stages survival fairness explain

In [ ]:
# 3/6  Bootstrap optimism correction (~10 min)
!python scripts/run_development.py --stages optimism

In [ ]:
# 4/6  DeepSurv and sensitivity analyses: piecewise Cox, nested CV (~10 min)
!python scripts/run_development.py --stages deepsurv sensitivity

In [ ]:
# 5/6  Simulation study (~30-60 min)
!python scripts/run_development.py --stages simulation

In [ ]:
# 6/6  Final models for notebook 02 + results summary (~2 min)
!python scripts/run_development.py --stages export summary

In [ ]:
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.max_columns', 40)
def table(name): return pd.read_csv(P.tables / f'{name}.csv')
def figure(name): display(Image(filename=str(P.figures / f'{name}.png'), width=900))

## Table 1 — baseline characteristics by outcome

In [ ]:
table('table1_cohort_by_outcome')

## Target leakage of follow-up time (Experiments A–E)

In [ ]:
figure('fig1_time_leakage')
table('leakage_BCD_paired_tests')

In [ ]:
table('leakage_E_horizon_summary')

## Replication of Chicco & Jurman (2020)

In [ ]:
table('replication_delta_mcc_S1_minus_S2')

## Survival analysis

In [ ]:
table('survival_cv_summary')[['model','harrell_c','uno_c','auc_90d','auc_180d','brier_90d','ibs','ibs_km_null']].round(3)

In [ ]:
figure('fig3_survival_cindex'); figure('fig2a_km_tertiles'); figure('fig2b_cox_forest')
table('table2_cox_hazard_ratios').round(3)

In [ ]:
table('table_s_ph_schoenfeld').round(3)

## Model comparison, calibration, clinical utility, sample size

In [ ]:
display(table('survival_paired_comparison').round(3)); display(table('calibration_oof_horizon_summary').round(3))
figure('fig6_decision_curve_90d'); figure('fig_s_calibration_oof_90d')
table('sample_size_riley')

## Simulation: leakage is structural

In [ ]:
figure('fig_s_simulation_leakage')
table('simulation_summary').round(3)

## Explanation and subgroup performance (out-of-fold only)

In [ ]:
figure('fig4_rsf_importance'); figure('fig4b_rsf_shap_beeswarm')
table('table3_subgroup_cindex').round(3)

## Auto-generated results summary

In [ ]:
from IPython.display import Markdown
Markdown((P.results / 'RESULTS_SUMMARY.md').read_text())